---
# Preparación de los datos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path().resolve().parent

In [2]:
df = pd.read_csv(BASE_DIR / 'data' / 'raw.csv', header=1)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_AMT3        

---

En el presente trabajo, se utiliza el conjunto de datos **Default of Credit Card Clients** con el objetivo de predecir si un cliente incumplirá su pago el próximo mes (`default payment next month`). Las variables predictoras disponibles son:

* **Datos demográficos:** Género (`SEX`), Educación (`EDUCATION`), Estado civil (`MARRIAGE`) y Edad (`AGE`).
* **Historial de pagos (`PAY_0` a `PAY_6`):** El estado de reembolso en los últimos 6 meses (retrasos en meses).
* **Estado de cuenta (`BILL_AMT1` a `BILL_AMT6`):** El monto de la factura mensual.
* **Pagos realizados (`PAY_AMT1` a `PAY_AMT6`):** El monto pagado en los meses anteriores.
* **Límite de crédito:** `LIMIT_BAL`.

Para una mejor comprensión del contexto de los datos, se han renombrado las columnas:

| Nombre original              | Nombre descriptivo            |
|------------------------------|-------------------------------|
| default payment next month   | DEFAULT                       |
| PAY_0                        | comportamiento_septiembre     |
| PAY_2 – PAY_6                | comportamiento_agosto – abril |
| BILL_AMT1                    | estado_cuenta_septiembre      |
| BILL_AMT2 – BILL_AMT6        | estado_cuenta_agosto – abril  |
| PAY_AMT1                     | abono_septiembre              |
| PAY_AMT2 – PAY_AMT6          | abono_agosto – abril          |
| LIMIT_BAL                    | linea_credito                 |
| SEX, EDUCATION, MARRIAGE, AGE| genero, educacion, estado_civil, edad |

In [3]:
df.rename(columns={
    # Target
    "default payment next month": "DEFAULT",
    # PAY
    "PAY_0": "comportamiento_septiembre",
    "PAY_2": "comportamiento_agosto",
    "PAY_3": "comportamiento_julio",
    "PAY_4": "comportamiento_junio",
    "PAY_5": "comportamiento_mayo",
    "PAY_6": "comportamiento_abril",
    # BILL_AMT
    "BILL_AMT1": "estado_cuenta_septiembre",
    "BILL_AMT2": "estado_cuenta_agosto",
    "BILL_AMT3": "estado_cuenta_julio",
    "BILL_AMT4": "estado_cuenta_junio",
    "BILL_AMT5": "estado_cuenta_mayo",
    "BILL_AMT6": "estado_cuenta_abril",
    # PAY_AMT
    "PAY_AMT1": "abono_septiembre",
    "PAY_AMT2": "abono_agosto",
    "PAY_AMT3": "abono_julio",
    "PAY_AMT4": "abono_junio",
    "PAY_AMT5": "abono_mayo",
    "PAY_AMT6": "abono_abril",
    # Demográficas
    "LIMIT_BAL": "linea_credito",
    "SEX": "genero",
    "EDUCATION": "educacion",
    "MARRIAGE": "estado_civil",
    "AGE": "edad"
}, inplace=True)

La siguiente tabla de frecuencias muestra la distribución del comportamiento de pagos a lo largo de los seis meses registrados. Los patrones observados se analizan en detalle en el notebook de definición del negocio.

In [4]:
comportamiento_cols = [
    'comportamiento_abril',
    'comportamiento_mayo',
    'comportamiento_junio',
    'comportamiento_julio',
    'comportamiento_agosto',
    'comportamiento_septiembre'
]

all_labels = sorted(set().union(*[df[col].dropna().unique() for col in comportamiento_cols]))

frecuencias = df[comportamiento_cols].apply(lambda col: col.value_counts().reindex(all_labels, fill_value=0)).T

frecuencias

,-2,-1,0,1,2,3,4,5,6,7,8
comportamiento_abril,4895,5740,16286,0,2766,184,49,13,19,46,2
comportamiento_mayo,4546,5539,16947,0,2626,178,84,17,4,58,1
comportamiento_junio,4348,5687,16455,2,3159,180,69,35,5,58,2
comportamiento_julio,4085,5938,15764,4,3819,240,76,21,23,27,3
comportamiento_agosto,3782,6050,15730,28,3927,326,99,25,12,20,1
comportamiento_septiembre,2759,5686,14737,3688,2667,322,76,26,11,9,19


La tabla de frecuencias también revela que los retrasos mayores a 2 meses tienen muy pocos casos individualmente, por lo que representan ruido estadístico. Además, desde el punto de vista del negocio, todos son igualmente severos al superar los 90 días de atraso. Por esta razón, se agrupan en una sola categoría (3).

In [5]:
for col in comportamiento_cols: df[col] = df[col].apply(lambda x: 3 if x >= 4 else x)


Después de analizar la frecuencia de clases en las variables demográficas, se agruparon categorías con muy pocos casos para reducir ruido estadístico:

- **genero**: recodificado a binario (1 = hombre, 0 = mujer).
- **educacion**: las categorías 0, 4, 5 y 6 se agruparon en "otros" (3), y se aplicó one-hot encoding generando dos variables binarias: `graduado` y `universitario`. La categoría "otros" actúa como referencia.
- **estado_civil**: las categorías 2 y 3 se agruparon en "no casado" (0), quedando binaria (1 = casado, 0 = no casado).

In [6]:
df = df.drop(columns='ID')
df["genero"] = df["genero"].replace(2, 0)
df['educacion'] = df['educacion'].replace({0: 3, 4: 3, 5: 3, 6: 3})
df["estado_civil"] = df["estado_civil"].replace({2: 0, 3: 0})

In [7]:
df['educacion'] = pd.Categorical(df['educacion'], categories=[3, 1, 2])
df = pd.get_dummies(df, columns=['educacion'], drop_first=True, dtype=int)


In [8]:
df['DEFAULT'].value_counts()

DEFAULT
0    23364
1     6636
Name: count, dtype: int64

Para garantizar la validez del modelo, se realizó un split estratificado que conserva la proporción de clases en ambos conjuntos. El split se aplica en esta etapa, antes de cualquier transformación, para evitar data leakage durante el entrenamiento. El modelo se entrenará con el 70% de los datos y el 30% restante se reservará para la evaluación final.

In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df['DEFAULT']
)

In [10]:
df.to_csv(BASE_DIR / 'data' / 'datos_procesados.csv', index=False)
train_df.to_csv(BASE_DIR / 'data' / 'datos_entrenamiento.csv', index=False)
test_df.to_csv(BASE_DIR / 'data' / 'datos_prueba.csv', index=False)